# Core 09 - Multi Agentic System

Objetivo: coordinar dos Agents dentro del mismo System y componer exclusivamente sus `RunResult` reales.

## Parametros de la demostracion

| Variable | Default | Proposito |
|---|---|---|
| AGENTIC_SYSTEMS_MULTI_PROVIDER | python-runtime | Compartir provider entre agentes. |
| symbols | tool y graph | Distribuir entradas reales por especialista. |
| aggregation | compose_result | Unir RunResult sin fabricar evidencia. |

## 1) Runtime compartido

El provider es seleccionable mediante `AGENTIC_SYSTEMS_MULTI_PROVIDER`; el default local permite ejecutar todo el notebook sin credenciales.

In [ ]:
import os

import agentic_systems as toolkit

PROVIDER = "python-runtime"
runtime = toolkit.runtime(provider=PROVIDER)
system = toolkit.system(runtime=runtime)
toolkit.show_json(runtime.describe(), title="Multi-system runtime")

## 2) Tools y Agents con responsabilidades separadas

In [ ]:
@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    return {"symbol": symbol, "is_public": symbol in toolkit.__all__}

@toolkit.tool
def review_observation(symbol: str, is_public: bool) -> dict:
    return {
        "symbol": symbol,
        "accepted": isinstance(is_public, bool),
        "observed_public_value": is_public,
    }

inspector = system.agent(
    name="multi_system_inspector",
    instructions="Inspecciona el simbolo con la Tool requerida.",
    tools=[inspect_public_api], runtime=runtime,
    contract=toolkit.AgentContract(must_call=["inspect_public_api"]),
)
reviewer = system.agent(
    name="multi_system_reviewer",
    instructions="Revisa la observacion con la Tool requerida.",
    tools=[review_observation], runtime=runtime,
    contract=toolkit.AgentContract(must_call=["review_observation"]),
)

## 3) Ejecutar y pasar evidencia

El Reviewer recibe campos normalizados del primer `RunResult`; no conoce una respuesta esperada.

In [ ]:
symbol = os.getenv("AGENTIC_SYSTEMS_DEMO_SYMBOL", "agent")
inspect_request = (
    {"tool": "inspect_public_api", "input": {"symbol": symbol}}
    if PROVIDER == "python-runtime"
    else f"Usa inspect_public_api para verificar {symbol}."
)
inspection_run = inspector.run(inspect_request, mode="eval")
inspection_output = toolkit.agent_output(inspection_run)
fields = inspection_output["fields"]

review_request = (
    {"tool": "review_observation", "input": {"symbol": fields["symbol"], "is_public": fields["is_public"]}}
    if PROVIDER == "python-runtime"
    else f"Usa review_observation con symbol={fields['symbol']} e is_public={fields['is_public']}."
)
review_run = reviewer.run(review_request, mode="eval")
review_output = toolkit.agent_output(review_run)

result = toolkit.compose_result(
    text=f"Inspeccion y revision completadas para {symbol}.",
    data={"inspection": inspection_output, "review": review_output},
    results=[inspection_run, review_run],
    mode="multi-agent",
    input={"symbol": symbol},
)

toolkit.human_result(result, title="Multi-agent RunResult", show_lineage=True)
toolkit.show_json({"inspection": inspection_output, "review": review_output}, title="Agent outputs")

## 4) API realmente ejercitada

In [ ]:
api_coverage = [
    "toolkit.runtime", "toolkit.system", "toolkit.tool", "system.agent",
    "Agent.run", "toolkit.agent_output", "toolkit.compose_result",
    "toolkit.human_result", "toolkit.show_json",
    "toolkit.AgentContract",
]
toolkit.show_json(api_coverage, title="Multi-system API coverage")

## Resultado esperado

Dos RunResult internos y uno compuesto. Runtime, usage, validation y tool events proceden de las ejecuciones observadas.